In [52]:
# import libraries
import pandas as pd
import re

print(pd.__version__)

3.0.1


In [53]:
# load the pokemon dataset
pokemon = pd.read_csv('https://raw.githubusercontent.com/sjsu-cs122-s26/122_datasets/refs/heads/main/Pokemon.csv')
print(pokemon.head())



   #                   Name Type 1  Type 2  Total  HP  Attack  Defense  \
0  1              Bulbasaur  Grass  Poison    318  45      49       49   
1  2                Ivysaur  Grass  Poison    405  60      62       63   
2  3               Venusaur  Grass  Poison    525  80      82       83   
3  3  VenusaurMega Venusaur  Grass  Poison    625  80     100      123   
4  4             Charmander   Fire     NaN    309  39      52       43   

   Sp. Atk  Sp. Def  Speed  Generation  Legendary  
0       65       65     45           1      False  
1       80       80     60           1      False  
2      100      100     80           1      False  
3      122      120     80           1      False  
4       60       50     65           1      False  


In [54]:
# dataset schema / baseline stats
print('Row count:', len(pokemon))
print('Column count:', len(pokemon.columns))
print('Columns:', pokemon.columns.tolist())
print('Unique pokemon names:', pokemon['Name'].nunique())
print('\nGeneration count:\n', pokemon['Generation'].value_counts().sort_index())
print('\nType 1 count (top 10):\n', pokemon['Type 1'].value_counts().head(10))
print('\nType 2 count (top 10):\n', pokemon['Type 2'].value_counts().head(10))
print('\nMissing values per column:\n', pokemon.isna().sum())

# try to

Row count: 800
Column count: 13
Columns: ['#', 'Name', 'Type 1', 'Type 2', 'Total', 'HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Generation', 'Legendary']
Unique pokemon names: 800

Generation count:
 Generation
1    166
2    106
3    160
4    121
5    165
6     82
Name: count, dtype: int64

Type 1 count (top 10):
 Type 1
Water       112
Normal       98
Grass        70
Bug          69
Psychic      57
Fire         52
Electric     44
Rock         44
Ground       32
Ghost        32
Name: count, dtype: int64

Type 2 count (top 10):
 Type 2
Flying      97
Ground      35
Poison      34
Psychic     33
Fighting    26
Grass       25
Fairy       23
Steel       22
Dark        20
Dragon      18
Name: count, dtype: int64

Missing values per column:
 #               0
Name            0
Type 1          0
Type 2        386
Total           0
HP              0
Attack          0
Defense         0
Sp. Atk         0
Sp. Def         0
Speed           0
Generation      0
Legendary       0
dtype

In [55]:
# clean text columns
text_cols = ['Name', 'Type 1', 'Type 2']
for col in text_cols:
    if col in pokemon.columns:
        pokemon[col] = pokemon[col].astype(str).str.strip()

In [56]:
# fill missing secondary type
if 'Type 2' in pokemon.columns:
    pokemon['Type 2'] = pokemon['Type 2'].replace('nan', pd.NA)
    pokemon['Type 2'] = pokemon['Type 2'].fillna('None')

In [57]:
#just to make sure all values are numeric
numeric_cols = ['#', 'HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Generation', 'Legendary']
for col in numeric_cols:
    if col in pokemon.columns:
        pokemon[col] = pd.to_numeric(pokemon[col], errors='coerce')

In [58]:
# recompute total stats and compare to Total column
stat_cols = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']
pokemon['Computed Total'] = pokemon[stat_cols].sum(axis=1)

if 'Total' in pokemon.columns:
    pokemon['Total Match'] = pokemon['Total'] == pokemon['Computed Total']

print(pokemon[['Name', 'Total', 'Computed Total', 'Total Match']].head())

                    Name  Total  Computed Total  Total Match
0              Bulbasaur    318             318         True
1                Ivysaur    405             405         True
2               Venusaur    525             525         True
3  VenusaurMega Venusaur    625             625         True
4             Charmander    309             309         True


In [59]:
# create features for later scoring
pokemon['Offense Score'] = pokemon[['Attack', 'Sp. Atk']].mean(axis=1)
pokemon['Defense Score'] = pokemon[['Defense', 'Sp. Def']].mean(axis=1)
pokemon['Bulk Score'] = pokemon[['HP', 'Defense', 'Sp. Def']].mean(axis=1)
pokemon['Mobility Score'] = pokemon['Speed']
pokemon['Best Attack'] = pokemon[['Attack', 'Sp. Atk']].max(axis=1)
pokemon['Best Defense'] = pokemon[['Defense', 'Sp. Def']].max(axis=1)

In [60]:
# normalize main scoring features for comparison
score_cols = ['Offense Score', 'Defense Score', 'Bulk Score', 'Mobility Score']
for col in score_cols:
    pokemon[col + ' Norm'] = (pokemon[col] - pokemon[col].min()) / (pokemon[col].max() - pokemon[col].min())

In [61]:
# simple viability score for early baseline
pokemon['Viability Score'] = (
    0.35 * pokemon['Offense Score Norm'] +
    0.25 * pokemon['Defense Score Norm'] +
    0.20 * pokemon['Bulk Score Norm'] +
    0.20 * pokemon['Mobility Score Norm']
)

pokemon[['Name', 'Type 1', 'Type 2', 'Viability Score']].sort_values(
    by='Viability Score',
    ascending=False
).head(10)

,Name,Type 1,Type 2,Viability Score
164,MewtwoMega Mewtwo Y,Psychic,None,0.689235
163,MewtwoMega Mewtwo X,Psychic,Fighting,0.688567
426,RayquazaMega Rayquaza,Dragon,Flying,0.687407
422,KyogrePrimal Kyogre,Water,None,0.679316
424,GroudonPrimal Groudon,Ground,Fire,0.679316
796,DiancieMega Diancie,Rock,Fairy,0.635208
418,LatiasMega Latias,Dragon,Psychic,0.621291
413,MetagrossMega Metagross,Steel,Psychic,0.620825
552,Arceus,Normal,None,0.619951
420,LatiosMega Latios,Dragon,Psychic,0.618960


In [62]:
# assign basic battle role
def assign_role(row):
    if row['Speed'] >= 100 and row['Best Attack'] >= 100:
        return 'DPS'
    elif row['HP'] >= 90 and row['Best Defense'] >= 100:
        return 'Tank'
    elif row['Best Attack'] >= 110:
        return 'Heavy hitter'
    elif row['Speed'] >= 110:
        return 'Fast'
    else:
        return 'Balanced'

pokemon['Role'] = pokemon.apply(assign_role, axis=1)
pokemon[['Name', 'Role']].head()

,Name,Role
0,Bulbasaur,Balanced
1,Ivysaur,Balanced
2,Venusaur,Balanced
3,VenusaurMega Venusaur,Heavy hitter
4,Charmander,Balanced


In [63]:
# try to decompose the pokemon into its base and its form
pattern = re.compile(r'[a-z][A-Z0-9%]')

# given the DF, get a set of the base name of the pokemons
def extract_base_name(df):
  seen = set()
  for _, group in df.groupby('#'):
    names = group['Name'].tolist()
    cleaned_names = [name for name in names if not pattern.search(name)]

    # single pokemon name?
    if cleaned_names:
      seen.update(cleaned_names)
    else:
      # the pokemon name is joined with its form
      # take the first one and infer the base
      first = names[0]
      m = pattern.search(first)
      cleaned_name = first[:m.start() + 1] if m else first # GiratinaAltered Forme -> Giratina
      seen.add(cleaned_name)
  return seen

# decompose the pokemon into its name and form
def parse_pokemon_name(name, base_names):
  if name in base_names:
    return name, 'Base'

  # split at the first occurence of an uppercase character
  m = re.search(r'(?<=[a-z])(?=[A-Z0-9%])', name)
  if m:
    idx = m.start()
    return name[:idx], name[idx:]

base_names = extract_base_name(pokemon)
tmp = pokemon['Name'].map(lambda name: parse_pokemon_name(name, base_names))
# add a new base_name column to the df
pokemon.insert(pokemon.columns.get_loc('Name') + 1, 'base_name', tmp.map(lambda x: x[0]))
# add a new form column to the df
pokemon.insert(pokemon.columns.get_loc('base_name') + 1, 'form', tmp.map(lambda x: x[1]))

In [64]:
# https://pokemondb.net/type

a = 0 # no effect
b = 0.5 # not very effective
c = 1 # normal
d = 2.0 # super-effective

# column = defense
# row = attack
effectiveness_matrix = [
  # normal
  [c, c, c, c, c, c, c, c, c, c, c, c, b, a, c, c, b, c],
  # fire
  [c, b, b, c, d, d, c, c, c, c, c, d, b, c, b, c, d, c],
  # water
  [c, d, b, c, b, c, c, c, d, c, c, c, d, c, b, c, c, c],
  # electric
  [c, c, d, b, b, c, c, c, a, d, c, c, c, c, b, c, c, c],
  # grass
  [c, b, d, c, b, c, c, b, d, b, c, b, d, c, b, c, b, c],
  # ice
  [c, b, b, c, d, b, c, c, d, d, c, c, c, c, d, c, b, c],
  # fighting
  [d, c, c, c, c, d, c, b, c, b, b, b, d, a, c, d, d, b],
  # poison
  [c, c, c, c, d, c, c, b, b, c, c, c, b, b, c, c, a, d],
  # ground
  [c, d, c, d, b, c, c, d, c, a, c, b, d, c, c, c, d, c],
  # flying
  [c, c, c, b, d, c, d, c, c, c, c, d, b, c, c, c, b, c],
  # psychic
  [c, c, c, c, c, c, d, d, c, c, b, c, c, c, c, a, b, c],
  # bug
  [c, b, c, c, d, c, b, b, c, b, d, c, c, b, c, d, b, b],
  # rock
  [c, d, c, c, c, d, b, c, b, d, c, d, c, c, c, c, b, c],
  # ghost
  [a, c, c, c, c, c, c, c, c, c, d, c, c, d, c, b, c, c],
  # dragon
  [c, c, c, c, c, c, c, c, c, c, c, c, c, c, d, c, b, a],
  # dark
  [c, c, c, c, c, c, b, c, c, c, d, c, c, d, c, b, c, b],
  # steel
  [c, b, b, b, c, d, c, c, c, c, c, c, d, c, c, c, b, d],
  # fairy
  [c, b, c, c, c, c, d, b, c, c, c, c, c, c, d, d, b, c]
]

for row in effectiveness_matrix:
  assert(len(row) == 18)

# types = {'normal','fire','water','electric','grass','ice','fighting','poison','ground','flying','psychic','bug','rock','ghost','dragon','dark','steel','fairy'}
# types_from_data = set()

# for a in pokemon['Type 1']:
#   types_from_data.add(a.lower())
# for b in pokemon['Type 2']:
#   types_from_data.add(b.lower())

# print('missing :: ', types_from_data - types)


In [ ]:
POKEMON_TYPES = [
    'Normal', 'Fire', 'Water', 'Electric', 'Grass', 'Ice', 'Fighting', 'Poison',
    'Ground', 'Flying', 'Psychic', 'Bug', 'Rock', 'Ghost', 'Dragon', 'Dark',
    'Steel', 'Fairy'
]
type_to_idx = {t: i for i, t in enumerate(POKEMON_TYPES)}

def normalize_type_name(t):
    # for Pokemon that have no secondary type
    if pd.isna(t):
        return None
    t = str(t).strip().title()
    return t if t in type_to_idx else None


def get_type_tuple_from_row(row):
    t1 = normalize_type_name(row['Type 1'])
    t2 = normalize_type_name(row['Type 2'])
    if t1 is None:
        return tuple()
    if t2 is None or t2 == t1:
        return (t1,)
    return tuple(sorted((t1, t2)))


def type_multiplier(attacking_type, defending_types):
    # the damage multipler of the attacker's type against the defender's types
    atk_idx = type_to_idx[attacking_type]
    mult = 1.0
    for defending_type in defending_types:
        mult *= effectiveness_matrix[atk_idx][type_to_idx[defending_type]]
    return mult


pokemon['Type Tuple'] = pokemon.apply(get_type_tuple_from_row, axis=1)
valid_type_tuples = pokemon['Type Tuple'][pokemon['Type Tuple'].map(len) > 0]

# distributions of type combinations that are more common in the dataset, for weighting offense and defense metrics
# more common type combinations should have more influence on the metrics since they might be more likely to be encountered
# in a battle
defender_combo_weights = valid_type_tuples.value_counts(normalize=True).to_dict()
attack_type_weights = valid_type_tuples.explode().value_counts(normalize=True).to_dict()

def compute_type_metrics(row):
    # type offense raw: expected damage multiplier when attacking, averaged across defenders
    # type defense raw: inverse of expected incoming damage
    # coverage super effective rate: % of opponents that can be hit super effectively (>1x) 
    # coverage resisted rate: % of opponents that resist the attacks (<1x)
    # incoming weakness rate: % of attacks that hit the pokemon for super effective damage (>1x)
    # incoming resistance rate: % of attacks that are resisted (<1x)
    # incoming immunity rate: % of attacks that have no effect (0x)
    # expected incoming multiplier: the weighted average of the damage multipler of all attacks against the pokemon's types
    stab_types = row['Type Tuple']
    if not stab_types:
        return pd.Series(
            {
                'Type Offense Raw': 0.0,
                'Type Defense Raw': 0.0,
                'Coverage Super Effective Rate': 0.0,
                'Coverage Resisted Rate': 0.0,
                'Incoming Weakness Rate': 0.0,
                'Incoming Resistance Rate': 0.0,
                'Incoming Immunity Rate': 0.0,
                'Expected Incoming Multiplier': 0.0,
            }
        )

    # STAB = same type attack bonus
    exp_best_stab = 0.0
    # super effective rate
    se_rate = 0.0
    # resisted rate
    resisted_rate = 0.0

    # simulate attacks against defense type combinations
    for defender_combo, w in defender_combo_weights.items():
        best = max(type_multiplier(atk_type, defender_combo) for atk_type in stab_types)
        exp_best_stab += w * best
        se_rate += w * (best > 1.0)
        resisted_rate += w * (best < 1.0)

    exp_incoming = 0.0
    weak_rate = 0.0
    resist_rate = 0.0
    immune_rate = 0.0

    # simulate attacks from all attack types against the pokemon's defense types
    for attack_type, w in attack_type_weights.items():
        incoming = type_multiplier(attack_type, stab_types)
        exp_incoming += w * incoming
        weak_rate += w * (incoming > 1.0)
        resist_rate += w * (0.0 < incoming < 1.0)
        immune_rate += w * (incoming == 0.0)

    # divide by 1e-9 to avoid division by zero
    type_defense_raw = 1.0 / (exp_incoming + 1e-9)

    return pd.Series(
        {
            'Type Offense Raw': exp_best_stab,
            'Type Defense Raw': type_defense_raw,
            'Coverage Super Effective Rate': se_rate,
            'Coverage Resisted Rate': resisted_rate,
            'Incoming Weakness Rate': weak_rate,
            'Incoming Resistance Rate': resist_rate,
            'Incoming Immunity Rate': immune_rate,
            'Expected Incoming Multiplier': exp_incoming,
        }
    )


def min_max_norm(series):
    low, high = series.min(), series.max()
    if high == low:
        return pd.Series(0.0, index=series.index)
    return (series - low) / (high - low)


type_metrics = pokemon.apply(compute_type_metrics, axis=1)
pokemon = pd.concat([pokemon, type_metrics], axis=1)

pokemon['Type Offense Norm'] = min_max_norm(pokemon['Type Offense Raw'])
pokemon['Type Defense Norm'] = min_max_norm(pokemon['Type Defense Raw'])

# weights TBD
pokemon['Offense Capability'] = (
    0.60 * pokemon['Offense Score Norm'] +
    0.40 * pokemon['Type Offense Norm']
)

pokemon['Defense Capability'] = (
    0.35 * pokemon['Defense Score Norm'] +
    0.25 * pokemon['Bulk Score Norm'] +
    0.40 * pokemon['Type Defense Norm']
)

pokemon['Meta Viability Score'] = (
    0.40 * pokemon['Offense Capability'] +
    0.30 * pokemon['Defense Capability'] +
    0.15 * pokemon['Mobility Score Norm'] +
    0.15 * pokemon['Viability Score']
)

In [66]:
  # final cleaned preview
print('Final shape:', pokemon.shape)
print('\nRole counts:\n', pokemon['Role'].value_counts())
# pokemon.head()
pokemon

Final shape: (800, 43)

Role counts:
 Role
Balanced        525
Heavy hitter    106
DPS              95
Tank             55
Fast             19
Name: count, dtype: int64


,#,Name,base_name,form,Type 1,Type 2,Total,HP,Attack,Defense,...,Coverage Resisted Rate,Incoming Weakness Rate,Incoming Resistance Rate,Incoming Immunity Rate,Expected Incoming Multiplier,Type Offense Norm,Type Defense Norm,Offense Capability,Defense Capability,Meta Viability Score
0,1,Bulbasaur,Bulbasaur,Base,Grass,Poison,318,45,49,49,...,0.11000,0.241351,0.299835,0.000000,1.071870,0.617722,0.356932,0.412971,0.262100,0.312959
1,2,Ivysaur,Ivysaur,Base,Grass,Poison,405,60,62,63,...,0.11000,0.241351,0.299835,0.000000,1.071870,0.617722,0.356932,0.462383,0.312811,0.373467
2,3,Venusaur,Venusaur,Base,Grass,Poison,525,80,82,83,...,0.11000,0.241351,0.299835,0.000000,1.071870,0.617722,0.356932,0.532971,0.382338,0.457228
3,3,VenusaurMega Venusaur,Venusaur,Mega Venusaur,Grass,Poison,625,80,100,123,...,0.11000,0.241351,0.299835,0.000000,1.071870,0.617722,0.356932,0.603559,0.468335,0.527123
4,4,Charmander,Charmander,Base,Fire,None,309,39,52,43,...,0.30750,0.206755,0.294893,0.000000,1.059308,0.348523,0.371681,0.301762,0.234242,0.276552
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,719,Diancie,Diancie,Base,Rock,Fairy,600,50,100,150,...,0.04500,0.277595,0.321252,0.041186,1.156507,0.781435,0.265911,0.630221,0.495332,0.518638
796,719,DiancieMega Diancie,Diancie,Mega Diancie,Rock,Fairy,700,50,160,110,...,0.04500,0.277595,0.321252,0.041186,1.156507,0.781435,0.265911,0.841986,0.380669,0.636276
797,720,HoopaHoopa Confined,Hoopa,Hoopa Confined,Psychic,Ghost,600,80,110,60,...,0.05875,0.079901,0.125206,0.127677,1.049423,0.505485,0.383534,0.625724,0.403012,0.503420
798,720,HoopaHoopa Unbound,Hoopa,Hoopa Unbound,Psychic,Dark,680,80,160,60,...,0.06000,0.092257,0.000000,0.074135,1.136738,0.486498,0.285958,0.741658,0.363982,0.559179


In [67]:
# write the new csv file
pokemon.to_csv('/tmp/pokemon_clean.csv', index=False)

#Visualizations
Currently some of the graphs base strength on total stats. We will need to make more visualizations based on the viability score of pokemon but these graphs can give good initial insight on the distribution of pokemon stats and how they may affect our viability score.
TBC


In [68]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(pokemon['Total'], bins=30) #Useful to show where the density of total stats lie, could potentially see if more total stats means better composition viability.
plt.title("Distribution of Pokemon Total Stats")

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
avgStatsPerType1 = pokemon.groupby("Type 1")[['Attack', "Defense", "Speed"]].mean()

#Useful to see which types specialize in which stats
avgStatsPerType1.plot(kind="bar")
plt.title("Average A/D/S Stat Per Type 1 of a Pokemon")
plt.ylabel("Stat #")
#Note: Should maybe find how to move the legend maybe outside of the plot

In [ ]:
sns.heatmap(pokemon[["HP", "Attack", "Defense", "Speed", "Sp. Atk", "Sp. Def", "Total"]].corr(), annot=True)
plt.title("Pokemon Stat Correlation Heatmap") # The higher the correlation number, the more correlated each stat is.
#This visualization helps find relationships, but most importantly shows the strongest contributors to the total stat. Potentially indicating viability.

In [ ]:
strongestPokemon = pokemon.nlargest(15, "Total") #"Strength" being currently based off Total stats, this could change.

sns.barplot(data=strongestPokemon, x="Total", y='Name')
plt.title("Top 15 Strongest Pokemon According to Total Stats")
plt.ylabel("Pokemon Name")
plt.xlabel("Pokemon Total Stats")
#This could help us find indications of strength. Just from this we can see a trend of Mega evolutions containing higher total stats than other pokemon.

In [ ]:
sns.boxplot(data=pokemon, x="Legendary", y="Total")
plt.title("Legendary Stat Total vs Non-Legendary Stat Totals")
plt.xlabel("Is Legendary?")
plt.ylabel("Stat Total")
#This plot shows us that on average, legendary pokemon have higher stat totals than regular pokemon.
#But there are outliers that exceed even legendary pokemon, my assumption being mega evolutions based on the past graph.